# 가장 간단한 그래프 · 조건부 엣지

강의 오후 ① 11 ~ 14장 (Academy 모듈 1 · L2 Simple Graph)

이번 시간에는 LangGraph 의 가장 기본 부품 세 가지로 아주 작은 그래프를 하나 만들어 보겠습니다. State, 노드, 엣지입니다.

예제는 앞에서 본 영어 튜터의 첫 단계입니다. 학생이 쓴 글이 일반 이메일이면 이메일 분석기로, IELTS 답안이면 IELTS 분석기로 보냅니다.

이번에는 LLM 을 쓰지 않습니다. 길을 고르는 것도 규칙 한 줄로 합니다. 그래프가 어떻게 생겼는지에만 집중하겠습니다.

먼저 LangGraph 를 불러옵니다. 버전은 2일차 저장소와 같게 맞춰 두었습니다.

In [ ]:
import "npm:@langchain/core@1.2.14";
import { Annotation, StateGraph, START, END } from "npm:@langchain/langgraph@1.4.18";

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

## ① State

먼저 State 를 정합니다. State 는 그래프 안에서 노드들이 같이 보는 칸입니다. 노드에서 노드로 건네지는 공책 한 장이라고 생각하시면 됩니다.

여기서는 칸이 두 개입니다. `text` 는 학생이 쓴 글이고, `log` 는 지금까지 거쳐 간 노드 이름을 적어 둘 칸입니다.

`log` 에는 reducer 가 붙어 있습니다. 노드가 새 값을 돌려주면 덮어쓰지 말고 뒤에 이어 붙이라는 규칙입니다. reducer 는 오후 ② 에서 자세히 다루니, 지금은 "이어 붙인다" 정도만 기억해 두시면 됩니다.

In [ ]:
// State: 노드끼리 주고받는 칸. 지금은 글(text)과, 거쳐 간 노드 기록(log) 두 칸
const State = Annotation.Root({
  text: Annotation<string>(),
  log: Annotation<string[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});

## ② 노드

다음은 노드입니다. 노드는 그냥 평범한 함수입니다. State 를 받아서, **바꾸고 싶은 칸만** 돌려줍니다.

`classify`, `analyze_email`, `analyze_ielts` 세 개를 만들겠습니다. 실제 분석 대신 지금은 무엇을 하는지 한 줄 찍고, `log` 에 자기 이름만 남깁니다.

한 가지 눈여겨보실 점은, 어느 노드도 "다음에 어디로 갈지" 를 모른다는 것입니다. 그건 노드가 아니라 엣지가 정합니다.

In [ ]:
// 노드: State 를 받아서, 바꾼 칸만 돌려주는 평범한 함수
function classify(state: typeof State.State) {
  console.log("▶ classify  어떤 글인지 본다");
  return { log: ["classify"] };
}
function analyzeEmail(state: typeof State.State) {
  console.log("▶ analyze_email  용건 · 격식 · 맺음말을 본다");
  return { log: ["analyze_email"] };
}
function analyzeIelts(state: typeof State.State) {
  console.log("▶ analyze_ielts  채점 기준(과제 수행 · 응집성 · 어휘 · 문법)으로 본다");
  return { log: ["analyze_ielts"] };
}

## ③ 조건부 엣지

이제 이 예제의 핵심인 조건부 엣지입니다. 조건부 엣지도 함수 하나입니다. State 를 보고, **다음에 갈 노드의 이름**을 돌려줍니다.

여기서는 규칙이 아주 단순합니다. 글이 Dear, Hi, Hello 로 시작하면 이메일로 보고 `analyze_email` 로, 아니면 `analyze_ielts` 로 보냅니다.

노드는 State 를 바꾸고, 조건부 엣지는 다음 갈 곳만 알려 줍니다. 둘 다 함수라는 점은 같습니다.

In [ ]:
// 조건부 엣지: State 를 보고 "다음에 갈 노드 이름" 을 돌려주는 함수
function decide(state: typeof State.State) {
  const isEmail = /^(dear|hi|hello)\b/i.test(state.text.trim());
  return isEmail ? "analyze_email" : "analyze_ielts";
}

## ④ 그래프로 잇기

부품이 다 모였으니 그래프로 이어 보겠습니다. `StateGraph` 에 방금 만든 State 를 넣고, `addNode` 로 노드 세 개를 올립니다.

선은 두 종류입니다.

- `addEdge` 는 실선입니다. 이 노드가 끝나면 무조건 저기로 갑니다. START 에서 classify 로, 두 분석기에서 END 로 가는 선이 실선입니다.
- `addConditionalEdges` 는 점선입니다. classify 가 끝나면 `decide` 에게 물어보고, 돌려준 이름의 노드로 갑니다. 세 번째 인자는 갈 수 있는 곳의 목록입니다.

`compile` 하면 LangGraph 가 빠진 선이 없는지 한 번 확인해 줍니다. 그 아래 두 줄은 그래프를 그림으로 띄우고(`showGraph`), 같은 그림을 mermaid 글자로도 찍어 줍니다. `showGraph` 는 맨 위 셀에 만들어 둔 작은 도우미로, 이후 노트북에서도 그래프를 만들 때마다 부릅니다.

In [ ]:
const graph = new StateGraph(State)
  .addNode("classify", classify)
  .addNode("analyze_email", analyzeEmail)
  .addNode("analyze_ielts", analyzeIelts)
  .addEdge(START, "classify")                                                  // 실선: 무조건
  .addConditionalEdges("classify", decide, ["analyze_email", "analyze_ielts"]) // 점선: decide 가 고른다
  .addEdge("analyze_email", END)
  .addEdge("analyze_ielts", END)
  .compile();
await showGraph(graph);

console.log(graph.getGraph().drawMermaid());

**돌려 보면** 그림에서 START → classify 와 두 분석기 → END 는 실선, classify 에서 두 분석기로 가는 선은 점선입니다. 글자로는 `__start__ --> classify` 가 실선(`-->`), `classify -.-> analyze_email` 과 `classify -.-> analyze_ielts` 가 점선(`-.->`)입니다. 앞 장표에서 그린 그림과 같습니다.

## ⑤ 실행

이제 실행해 보겠습니다. `invoke` 에 처음 State 를 넣으면 그래프가 START 부터 END 까지 끝까지 돕니다.

첫 번째는 Dear 로 시작하는 이메일, 두 번째는 IELTS 답안입니다. 같은 그래프인데 글에 따라 다른 길로 가는지 보세요.

In [ ]:
const email = await graph.invoke({ text: "Dear Mr. Kim, I am writing to ask about the schedule..." });
console.log("거쳐 간 길:", email.log.join(" → "));

const ielts = await graph.invoke({ text: "Some people believe that technology makes life easier..." });
console.log("거쳐 간 길:", ielts.log.join(" → "));

**돌려 보면** 이메일은 classify → analyze_email, IELTS 답안은 classify → analyze_ielts 로 갑니다. 노드마다 찍은 한 줄과, `log` 칸에 쌓인 이름으로 어느 길을 지났는지 확인할 수 있습니다.

정리하면 State 는 같이 보는 칸, 노드는 바꾼 칸만 돌려주는 함수, 엣지는 다음 갈 곳입니다.

지금은 길을 규칙이 골랐습니다. 이 자리를 LLM 에게 맡기면 무엇이 달라지는지는 뒤의 Router 에서 보겠습니다.